# CGH Pathology SAM 3.1 Strategy 2 - Full 41-Tile Dataset

Run this notebook from the `sam31-cgh-strategy2` repo root after cloning the full data branch separately.

Strategy 2 is now retargeted to the current 41-tile CellSeg1/CGH P2 dataset:

- Source dataset defaults to `~/Desktop/1.Data/training_pa_he_annotation_full/cellseg1_cgh_p2_combined_41_full`.
- `prepare_sam31_dataset.py` stages the source into the legacy SAM3 package layout and regenerates COCO JSON.
- Expected split is 37 train / 4 validation-test tiles.
- Validation/test tiles stay comparable with prior Strategy 2: `p2_tile_05`, `p2_tile_10`, `p2_tile_15`, and `p2_tile_20`.
- Training remains SAM3/SAM 3.1 full unfreeze by default with lower LR scale.
- Live plots parse SAM3 losses and meters into `strategy2_live_metrics.jsonl`.
- Every run writes to a new `outputs/strategy2_41tiles_*` folder and does not delete old outputs.


In [ ]:
from pathlib import Path
from collections import deque
import ast
import json
import os
import re
import subprocess
import sys
import time

PACKAGE_ROOT = Path.cwd().resolve()
assert (PACKAGE_ROOT / "prepare_sam31_dataset.py").exists(), f"Start Jupyter from the sam31-cgh-strategy2 repo root, got {PACKAGE_ROOT}"

DEFAULT_SOURCE_DATASET_ROOT = Path.home() / "Desktop/1.Data/training_pa_he_annotation_full/cellseg1_cgh_p2_combined_41_full"
SOURCE_DATASET_ROOT = Path(os.environ.get(
    "CGH_SAM31_SOURCE_ROOT",
    os.environ.get("CGH_DATASET_ROOT", DEFAULT_SOURCE_DATASET_ROOT),
)).expanduser().resolve()
os.environ["CGH_SAM31_SOURCE_ROOT"] = str(SOURCE_DATASET_ROOT)

SAM3_REPO = Path(os.environ.get("SAM3_REPO", PACKAGE_ROOT.parent / "sam3")).expanduser().resolve()
HF_MODEL_ID = "facebook/sam3"
DATASET_ROOT = PACKAGE_ROOT / "dataset"
COCO_ROOT = DATASET_ROOT / "coco_sam3" / "cgh_pathology_sam31"
RUN_TAG = os.environ.get("SAM31_RUN_TAG") or time.strftime("strategy2_41tiles_full_unfreeze_%Y%m%d_%H%M%S")
OUTPUT_ROOT = PACKAGE_ROOT / "outputs" / RUN_TAG

print("PACKAGE_ROOT:", PACKAGE_ROOT)
print("SOURCE_DATASET_ROOT:", SOURCE_DATASET_ROOT)
print("SOURCE exists:", SOURCE_DATASET_ROOT.exists())
print("SAM3_REPO:", SAM3_REPO)
print("COCO_ROOT:", COCO_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("Old run folders are not touched. Change SAM31_RUN_TAG or RUN_TAG before training if you want a specific folder name.")


## 1. Environment

Install PyTorch separately for the GPU assigned by the cluster. For the Blackwell RTX PRO 6000 node we used CUDA 12.8 wheels; for V100 use CUDA 11.8 wheels.

The dependency cell below intentionally defaults to dry-run mode. Enable it only if this Jupyter kernel is allowed to install packages.


In [ ]:
INSTALL_DEPS = False
INSTALL_SAM3_EDITABLE = False

if INSTALL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "--user", "-r", str(PACKAGE_ROOT / "requirements_sam31.txt")], check=True)

if INSTALL_SAM3_EDITABLE:
    if not SAM3_REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/facebookresearch/sam3.git", str(SAM3_REPO)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--user", "-e", ".[train]"], cwd=SAM3_REPO, check=True)

mods = ["numpy", "pandas", "PIL", "scipy", "pycocotools", "hydra", "submitit", "einops", "wandb", "torch"]
for name in mods:
    try:
        mod = __import__(name)
        version = getattr(mod, "__version__", "ok")
        print(f"{name}: {version}")
    except Exception as exc:
        print(f"{name}: MISSING - {type(exc).__name__}: {exc}")

try:
    import torch
    print("torch cuda:", torch.version.cuda)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu:", torch.cuda.get_device_name(0))
except Exception:
    pass


## 2. Verify Dataset

This stages the full 41-tile CellSeg1 dataset into the Strategy2 SAM3 layout, regenerates COCO JSON, and verifies the expected 37/4 split.


In [ ]:
assert SOURCE_DATASET_ROOT.exists(), f"Missing source dataset: {SOURCE_DATASET_ROOT}"
subprocess.run([sys.executable, str(PACKAGE_ROOT / "prepare_sam31_dataset.py")], check=True)

summary_path = DATASET_ROOT / "sam31_dataset_summary.json"
summary = json.loads(summary_path.read_text())
print(json.dumps(summary, indent=2))

expected = {"images": 41, "train_images": 37, "test_images": 4}
for key, value in expected.items():
    assert summary.get(key) == value, f"Expected {key}={value}, got {summary.get(key)}"

assert summary.get("staged_source", {}).get("copied_images") == 41
assert summary.get("staged_source", {}).get("copied_cell_masks") == 41


In [ ]:
import pandas as pd
from IPython.display import Markdown, display

manifest = pd.read_csv(DATASET_ROOT / "sam31_manifest.csv")
display(manifest.groupby(["split", "category", "include_for_training"]).size().rename("n").reset_index())

qc_path = DATASET_ROOT / "metadata" / "DATASET_QC.md"
if qc_path.exists():
    display(Markdown(qc_path.read_text()))


## 3. Visual Check

Use this before training. Blue is nucleus, green is trainable clear/compact cell boundary, red is stroma, orange is uncertain/ignore.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

tile_id = os.getenv("SAM31_PREVIEW_TILE_ID", "yolo_tile_23")

def read_mask_or_zero(path: Path, shape):
    if path.exists():
        return np.asarray(Image.open(path))
    return np.zeros(shape, dtype=np.uint8)

image = np.asarray(Image.open(DATASET_ROOT / "images" / f"{tile_id}.png").convert("RGB"))
cell_mask = np.asarray(Image.open(DATASET_ROOT / "cell_instance_masks" / f"{tile_id}.png"))
nucleus_mask = read_mask_or_zero(DATASET_ROOT / "auxiliary_masks" / f"{tile_id}_gt_nucleus_instances.png", cell_mask.shape)
stroma_mask = read_mask_or_zero(DATASET_ROOT / "auxiliary_masks" / f"{tile_id}_gt_stroma.png", cell_mask.shape) > 0
uncertain_mask = read_mask_or_zero(DATASET_ROOT / "auxiliary_masks" / f"{tile_id}_gt_uncertain_ignore.png", cell_mask.shape) > 0

overlay = image.astype(float) / 255.0
overlay[nucleus_mask > 0] = overlay[nucleus_mask > 0] * 0.55 + np.array([0.05, 0.25, 1.0]) * 0.45
overlay[cell_mask > 0] = overlay[cell_mask > 0] * 0.55 + np.array([0.0, 0.8, 0.25]) * 0.45
overlay[stroma_mask] = overlay[stroma_mask] * 0.55 + np.array([1.0, 0.1, 0.05]) * 0.45
overlay[uncertain_mask] = overlay[uncertain_mask] * 0.55 + np.array([1.0, 0.6, 0.0]) * 0.45

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(image)
axes[0].set_title("Raw")
axes[1].imshow(cell_mask, cmap="nipy_spectral")
axes[1].set_title(f"Cell instances: {int(cell_mask.max())}")
axes[2].imshow(overlay)
axes[2].set_title(f"Overlay: {tile_id}")
for ax in axes:
    ax.axis("off")
plt.show()


## 4. Verify Hugging Face Access

You need accepted access to `facebook/sam3` and a Hugging Face token on the cluster.


In [ ]:
try:
    from huggingface_hub import hf_hub_download, model_info
    info = model_info(HF_MODEL_ID)
    hf_hub_download(repo_id=HF_MODEL_ID, filename="config.json")
    print("HF model accessible:", info.modelId)
except Exception as exc:
    print("Could not verify HF access.")
    print("Run: hf auth login")
    print("For fine-grained tokens, enable read access to public gated repos.")
    print(type(exc).__name__, exc)


## 5. Create Strategy 2 SAM3 Config

Default Strategy 2 remains full unfreeze with lower LR scale. Override the environment variables below if memory/runtime becomes a problem.


In [ ]:
assert SAM3_REPO.exists(), f"SAM3 repo not found: {SAM3_REPO}"

FULL_UNFREEZE = os.getenv("SAM31_FULL_UNFREEZE", "1") != "0"
LR_SCALE = float(os.getenv("SAM31_LR_SCALE", "0.03" if FULL_UNFREEZE else "0.10"))
MAX_DATA_EPOCHS = int(os.getenv("SAM31_MAX_DATA_EPOCHS", "80"))
TARGET_EPOCH_SIZE = int(os.getenv("SAM31_TARGET_EPOCH_SIZE", "1000"))
VAL_EPOCH_FREQ = int(os.getenv("SAM31_VAL_EPOCH_FREQ", "5"))
TRAIN_BATCH_SIZE = int(os.getenv("SAM31_TRAIN_BATCH_SIZE", "1"))
VAL_BATCH_SIZE = int(os.getenv("SAM31_VAL_BATCH_SIZE", "1"))
GRADIENT_ACCUMULATION_STEPS = int(os.getenv("SAM31_GRADIENT_ACCUMULATION_STEPS", "1"))

config_proc = subprocess.run(
    [sys.executable, str(PACKAGE_ROOT / "write_sam3_config.py"), "--sam3-repo", str(SAM3_REPO)],
    check=True,
    capture_output=True,
    text=True,
)
CONFIG_PATH = Path(config_proc.stdout.strip())
subprocess.run([sys.executable, str(PACKAGE_ROOT / "patch_sam3_cluster.py"), "--sam3-repo", str(SAM3_REPO)], check=True)

text = CONFIG_PATH.read_text()

def set_line(pattern, replacement, text):
    new_text, n = re.subn(pattern, replacement, text, count=1, flags=re.MULTILINE)
    if n == 0:
        print("WARNING: did not match", pattern)
    return new_text

text = set_line(r"^  roboflow_vl_100_root: .*$", f"  roboflow_vl_100_root: {json.dumps(str(DATASET_ROOT / 'coco_sam3'))}", text)
text = set_line(r"^  experiment_log_dir: .*$", f"  experiment_log_dir: {json.dumps(str(OUTPUT_ROOT))}", text)
text = set_line(r"^  lr_scale: .*$", f"  lr_scale: {LR_SCALE}", text)
text = set_line(r"^  max_data_epochs: .*$", f"  max_data_epochs: {MAX_DATA_EPOCHS}", text)
text = set_line(r"^  target_epoch_size: .*$", f"  target_epoch_size: {TARGET_EPOCH_SIZE}", text)
text = set_line(r"^  train_batch_size: .*$", f"  train_batch_size: {TRAIN_BATCH_SIZE}", text)
text = set_line(r"^  val_batch_size: .*$", f"  val_batch_size: {VAL_BATCH_SIZE}", text)
text = set_line(r"^  gradient_accumulation_steps: .*$", f"  gradient_accumulation_steps: {GRADIENT_ACCUMULATION_STEPS}", text)
text = set_line(r"^  val_epoch_freq: .*$", f"  val_epoch_freq: {VAL_EPOCH_FREQ}", text)
CONFIG_PATH.write_text(text)

CONFIG_NAME = "configs/cgh_pathology/cgh_pathology_sam31_seg.yaml"
print("Wrote:", CONFIG_PATH)
print("Use config name:", CONFIG_NAME)
print("FULL_UNFREEZE:", FULL_UNFREEZE)
print("LR_SCALE:", LR_SCALE)
print("MAX_DATA_EPOCHS:", MAX_DATA_EPOCHS)


In [ ]:
config_text = CONFIG_PATH.read_text()
for token in [
    "roboflow_vl_100_root:",
    "experiment_log_dir:",
    "enable_segmentation:",
    "resolution:",
    "max_ann_per_img:",
    "max_train_queries:",
    "max_val_queries:",
    "max_data_epochs:",
    "target_epoch_size:",
    "train_batch_size:",
    "val_batch_size:",
    "val_epoch_freq:",
    "lr_scale:",
    "lr_transformer:",
    "lr_vision_backbone:",
    "lr_language_backbone:",
    "skip_saving_ckpts:",
    "amp_dtype:",
]:
    for line in config_text.splitlines():
        if token in line:
            print(line)
            break

loss_lines = [line.strip() for line in config_text.splitlines() if "SemanticSegCriterion" in line]
print("Loss target(s):")
for line in loss_lines:
    print(" ", line)


## 6. Check Running Training Processes

Use this to avoid accidentally running two training jobs on the same GPU.


In [ ]:
subprocess.run(["bash", "-lc", "ps -ef | grep 'sam3/train/train.py' | grep -v grep || true"], check=True)
try:
    subprocess.run(["nvidia-smi"], check=False)
except FileNotFoundError:
    print("nvidia-smi not found in this environment")


## 7. Launch Training With Live Notebook Plots

Set `RUN_TRAINING = True` when ready. The cell streams SAM3 logs, parses key meters, saves `strategy2_live_metrics.jsonl`, and redraws plots in the notebook every few seconds.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import clear_output, display

RUN_TRAINING = False
ALLOW_REUSE_OUTPUT_ROOT = False
USE_CLUSTER = 0
NUM_GPUS = 1
NUM_NODES = 1
PARTITION = None
ACCOUNT = None
QOS = None

WANDB_MODE = os.environ.get("WANDB_MODE", "offline")
WANDB_RUN_NAME = os.environ.get("WANDB_RUN_NAME", RUN_TAG)

cmd = [
    sys.executable,
    str(SAM3_REPO / "sam3" / "train" / "train.py"),
    "-c",
    CONFIG_NAME,
    "--use-cluster",
    str(USE_CLUSTER),
    "--num-gpus",
    str(NUM_GPUS),
    "--num-nodes",
    str(NUM_NODES),
]
if PARTITION:
    cmd += ["--partition", PARTITION]
if ACCOUNT:
    cmd += ["--account", ACCOUNT]
if QOS:
    cmd += ["--qos", QOS]

print(" ".join(cmd))
print("WANDB_MODE:", WANDB_MODE)
print("FULL_UNFREEZE:", FULL_UNFREEZE)
print("SAM3_CGH_FREEZE_BACKBONES:", "0" if FULL_UNFREEZE else "1")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
live_log_path = OUTPUT_ROOT / "strategy2_live_training.log"
metrics_jsonl = OUTPUT_ROOT / "strategy2_live_metrics.jsonl"

metric_rows = []
log_tail = deque(maxlen=24)

def gpu_snapshot():
    try:
        out = subprocess.check_output(
            [
                "nvidia-smi",
                "--query-gpu=name,memory.used,memory.total,utilization.gpu,temperature.gpu",
                "--format=csv,noheader,nounits",
            ],
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
        return out
    except Exception:
        return "nvidia-smi unavailable"

def parse_metric_payload(line):
    if "Losses and meters:" not in line and "Meters:" not in line:
        return None
    marker = "Losses and meters:" if "Losses and meters:" in line else "Meters:"
    payload = line.split(marker, 1)[1].strip()
    try:
        data = ast.literal_eval(payload)
    except Exception:
        return None
    row = {
        "time": time.time(),
        "epoch": data.get("Trainer/epoch"),
        "where": data.get("Trainer/where"),
        "steps_train": data.get("Trainer/steps_train"),
        "steps_val": data.get("Trainer/steps_val"),
        "train_loss": data.get("Losses/train_all_loss"),
        "train_core_loss": data.get("Losses/train_all_core_loss"),
        "train_semantic_miou": data.get("Losses/train_all_miou_semantic_seg"),
        "train_semantic_seg_loss": data.get("Losses/train_all_loss_semantic_seg"),
        "train_semantic_dice_loss": data.get("Losses/train_all_loss_semantic_dice"),
        "train_mask_loss": data.get("Losses/train_all_loss_mask"),
        "train_dice_loss": data.get("Losses/train_all_loss_dice"),
        "val_bbox_ap": data.get("Meters_train/val_roboflow100/detection/coco_eval_bbox_AP"),
        "val_bbox_ap50": data.get("Meters_train/val_roboflow100/detection/coco_eval_bbox_AP_50"),
        "val_bbox_ap75": data.get("Meters_train/val_roboflow100/detection/coco_eval_bbox_AP_75"),
    }
    return {k: v for k, v in row.items() if v is not None}

def parse_train_epoch_line(line):
    m = re.search(r"Train Epoch:\s*\[(\d+)\]\[\s*(\d+)/(\d+)\].*?Losses/train_all_loss:\s*([0-9.eE+-]+)", line)
    if not m:
        return None
    epoch, batch, total, loss = m.groups()
    return {
        "time": time.time(),
        "epoch": int(epoch),
        "batch": int(batch),
        "batches_per_epoch": int(total),
        "train_loss": float(loss),
    }

def append_metric(row):
    metric_rows.append(row)
    with metrics_jsonl.open("a") as f:
        f.write(json.dumps(row) + "\n")

def redraw(returncode=None):
    clear_output(wait=True)
    print("Command:", " ".join(cmd))
    print("Output:", OUTPUT_ROOT)
    print("GPU:", gpu_snapshot())
    if returncode is not None:
        print("Process return code:", returncode)

    if metric_rows:
        df = pd.DataFrame(metric_rows)
        x = df["epoch"] if "epoch" in df else pd.Series(range(len(df)))
        fig, axes = plt.subplots(1, 3, figsize=(17, 4))

        if "train_loss" in df:
            axes[0].plot(x, df["train_loss"], marker="o", linewidth=1)
            axes[0].set_title("Train loss")
            axes[0].set_xlabel("epoch")
            axes[0].grid(True, alpha=0.3)

        for col, label in [
            ("train_semantic_miou", "semantic mIoU"),
            ("val_bbox_ap", "val bbox AP"),
            ("val_bbox_ap50", "val bbox AP50"),
        ]:
            if col in df and df[col].notna().any():
                axes[1].plot(df.loc[df[col].notna(), "epoch"], df.loc[df[col].notna(), col], marker="o", linewidth=1, label=label)
        axes[1].set_title("Quality metrics")
        axes[1].set_xlabel("epoch")
        axes[1].grid(True, alpha=0.3)
        axes[1].legend()

        for col, label in [
            ("train_semantic_dice_loss", "semantic dice loss"),
            ("train_mask_loss", "mask loss"),
            ("train_dice_loss", "dice loss"),
        ]:
            if col in df and df[col].notna().any():
                axes[2].plot(df.loc[df[col].notna(), "epoch"], df.loc[df[col].notna(), col], marker="o", linewidth=1, label=label)
        axes[2].set_title("Segmentation losses")
        axes[2].set_xlabel("epoch")
        axes[2].grid(True, alpha=0.3)
        axes[2].legend()

        plt.tight_layout()
        display(fig)
        plt.close(fig)
        display(df.tail(8))
    else:
        print("No parsed metrics yet. Waiting for SAM3 logger lines.")

    print("\n--- log tail ---")
    print("\n".join(log_tail))

if RUN_TRAINING:
    if not ALLOW_REUSE_OUTPUT_ROOT and OUTPUT_ROOT.exists():
        existing = [
            OUTPUT_ROOT / "checkpoints" / "checkpoint.pt",
            OUTPUT_ROOT / "strategy2_live_training.log",
            OUTPUT_ROOT / "strategy2_live_metrics.jsonl",
        ]
        existing = [path for path in existing if path.exists()]
        if existing:
            raise RuntimeError(
                "OUTPUT_ROOT already contains training artifacts. "
                "Change RUN_TAG/SAM31_RUN_TAG for a new folder, or set "
                "ALLOW_REUSE_OUTPUT_ROOT=True if you intentionally want to continue/reuse it. "
                f"Existing: {existing}"
            )

    env = os.environ.copy()
    env.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    env.setdefault("PYTHONUNBUFFERED", "1")
    env["SAM3_CGH_FREEZE_BACKBONES"] = "0" if FULL_UNFREEZE else "1"
    env.setdefault("WANDB_PROJECT", "sam31-cgh")
    env["WANDB_RUN_NAME"] = WANDB_RUN_NAME
    env["WANDB_MODE"] = WANDB_MODE

    if metrics_jsonl.exists():
        metrics_jsonl.unlink()
    with live_log_path.open("w") as log_file:
        proc = subprocess.Popen(
            cmd,
            cwd=SAM3_REPO,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        last_redraw = 0.0
        for line in proc.stdout:
            line = line.rstrip("\n")
            log_file.write(line + "\n")
            log_file.flush()
            log_tail.append(line)
            row = parse_metric_payload(line) or parse_train_epoch_line(line)
            if row:
                append_metric(row)
            now = time.time()
            if row or now - last_redraw > 5:
                redraw()
                last_redraw = now
        returncode = proc.wait()
    redraw(returncode=returncode)
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, cmd)
else:
    print("Dry run only. Set RUN_TRAINING=True when ready.")


## 8. Reload Live Metrics

Use this after interruption or after training finishes to redraw the notebook curves from `strategy2_live_metrics.jsonl`.


In [ ]:
metrics_jsonl = OUTPUT_ROOT / "strategy2_live_metrics.jsonl"
if not metrics_jsonl.exists():
    print("No metrics JSONL yet:", metrics_jsonl)
else:
    rows = [json.loads(line) for line in metrics_jsonl.read_text().splitlines() if line.strip()]
    df = pd.DataFrame(rows)
    display(df.tail(10))

    fig, axes = plt.subplots(1, 3, figsize=(17, 4))
    x = df["epoch"] if "epoch" in df else pd.Series(range(len(df)))
    if "train_loss" in df:
        axes[0].plot(x, df["train_loss"], marker="o")
        axes[0].set_title("Train loss")
    for col, label in [("train_semantic_miou", "semantic mIoU"), ("val_bbox_ap", "val bbox AP"), ("val_bbox_ap50", "val bbox AP50")]:
        if col in df and df[col].notna().any():
            axes[1].plot(df.loc[df[col].notna(), "epoch"], df.loc[df[col].notna(), col], marker="o", label=label)
    axes[1].set_title("Quality metrics")
    axes[1].legend()
    for col, label in [("train_semantic_dice_loss", "semantic dice loss"), ("train_mask_loss", "mask loss"), ("train_dice_loss", "dice loss")]:
        if col in df and df[col].notna().any():
            axes[2].plot(df.loc[df[col].notna(), "epoch"], df.loc[df[col].notna(), col], marker="o", label=label)
    axes[2].set_title("Segmentation losses")
    axes[2].legend()
    for ax in axes:
        ax.grid(True, alpha=0.3)
        ax.set_xlabel("epoch")
    plt.tight_layout()
    plt.show()


## 9. TensorBoard In Notebook

This is optional. The live plot above is usually enough, but TensorBoard shows every scalar written by SAM3.


In [ ]:
TENSORBOARD_LOGDIR = str(OUTPUT_ROOT / "tensorboard")
print(TENSORBOARD_LOGDIR)

ip = get_ipython()
if ip is None:
    print("TensorBoard magic is only available inside IPython/Jupyter.")
else:
    ip.run_line_magic("load_ext", "tensorboard")
    ip.run_line_magic("tensorboard", f"--logdir {TENSORBOARD_LOGDIR}")


## 10. Checkpoint And Output Summary


In [ ]:
print("Output root:", OUTPUT_ROOT)
for path in [
    OUTPUT_ROOT / "checkpoints" / "checkpoint.pt",
    OUTPUT_ROOT / "strategy2_live_training.log",
    OUTPUT_ROOT / "strategy2_live_metrics.jsonl",
]:
    print(path, "exists=" + str(path.exists()), "size=" + (str(path.stat().st_size) if path.exists() else "NA"))

if (OUTPUT_ROOT / "checkpoints").exists():
    subprocess.run(["bash", "-lc", f"ls -lh {str(OUTPUT_ROOT / 'checkpoints')!r}"], check=False)


## 11. Qualitative Check After Training

Run this after SAM3 training finishes. It loads the trained checkpoint, runs text-prompted SAM3 inference on the COCO test images, and saves side-by-side `original / ground truth / prediction` PNGs.


In [ ]:
# Qualitative check after training: original vs ground truth vs SAM3 prediction.
from pathlib import Path
from contextlib import nullcontext
import csv

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Image as IPyImage, display
from PIL import Image
from pycocotools import mask as coco_mask

SAM31_COMPARE_LIMIT = int(os.getenv("SAM31_COMPARE_LIMIT", "4"))
SAM31_COMPARE_SPLIT = os.getenv("SAM31_COMPARE_SPLIT", "test")
SAM31_COMPARE_PROMPTS = [
    token.strip()
    for token in os.getenv("SAM31_COMPARE_PROMPTS", "clear cell boundary,compact cell boundary").split(",")
    if token.strip()
]
SAM31_COMPARE_GT_CATEGORIES = {
    token.strip()
    for token in os.getenv("SAM31_COMPARE_GT_CATEGORIES", "clear_cell_boundary,compact_cell_boundary").split(",")
    if token.strip()
}
SAM31_COMPARE_SCORE_THRESH = float(os.getenv("SAM31_COMPARE_SCORE_THRESH", "0.30"))
SAM31_COMPARE_MIN_AREA = int(os.getenv("SAM31_COMPARE_MIN_AREA", "20"))
SAM31_COMPARE_MASK_THRESH_RAW = os.getenv("SAM31_COMPARE_MASK_THRESH", "").strip()
SAM31_COMPARE_MASK_THRESH = float(SAM31_COMPARE_MASK_THRESH_RAW) if SAM31_COMPARE_MASK_THRESH_RAW else None

checkpoint_candidates = [
    Path(os.getenv("SAM31_COMPARE_CHECKPOINT", "")) if os.getenv("SAM31_COMPARE_CHECKPOINT") else None,
    OUTPUT_ROOT / "checkpoints" / "checkpoint.pt",
    OUTPUT_ROOT / "checkpoint.pt",
]
checkpoint_candidates.extend(sorted((OUTPUT_ROOT / "checkpoints").glob("*.pt")) if (OUTPUT_ROOT / "checkpoints").exists() else [])
SAM31_CHECKPOINT = next((path for path in checkpoint_candidates if path and path.exists()), None)
assert SAM31_CHECKPOINT is not None, f"No SAM3 checkpoint found under {OUTPUT_ROOT}. Train first, or set SAM31_COMPARE_CHECKPOINT=/path/to/checkpoint.pt"

compare_coco_path = COCO_ROOT / SAM31_COMPARE_SPLIT / "_annotations.coco.json"
assert compare_coco_path.exists(), compare_coco_path
compare_coco = json.loads(compare_coco_path.read_text())
image_infos = sorted(compare_coco["images"], key=lambda row: row["file_name"])[:SAM31_COMPARE_LIMIT]
assert image_infos, f"No images found in COCO split: {compare_coco_path}"

cat_by_id = {cat["id"]: cat["name"] for cat in compare_coco["categories"]}
anns_by_image = {}
for ann in compare_coco["annotations"]:
    anns_by_image.setdefault(ann["image_id"], []).append(ann)

COMPARE_DIR = OUTPUT_ROOT / f"sam31_comparison_{SAM31_COMPARE_SPLIT}_score{SAM31_COMPARE_SCORE_THRESH:.2f}"
COMPARE_DIR.mkdir(parents=True, exist_ok=True)
PRED_MASK_DIR = COMPARE_DIR / "pred_masks"
PRED_MASK_DIR.mkdir(exist_ok=True)

GT_COLOR = np.array([0, 180, 120], dtype=np.uint8)      # green in RGB
PRED_COLOR = np.array([255, 200, 0], dtype=np.uint8)    # amber in RGB


def decode_coco_segmentation(segmentation, height: int, width: int) -> np.ndarray:
    if isinstance(segmentation, dict):
        if isinstance(segmentation.get("counts"), list):
            rle = coco_mask.frPyObjects(segmentation, height, width)
        else:
            rle = segmentation
        return coco_mask.decode(rle).astype(bool)
    if isinstance(segmentation, list):
        rles = coco_mask.frPyObjects(segmentation, height, width)
        decoded = coco_mask.decode(rles)
        if decoded.ndim == 3:
            decoded = decoded.any(axis=2)
        return decoded.astype(bool)
    raise TypeError(f"Unsupported COCO segmentation type: {type(segmentation)!r}")


def coco_gt_instance_mask(image_info: dict) -> np.ndarray:
    height, width = int(image_info["height"]), int(image_info["width"])
    out = np.zeros((height, width), dtype=np.uint16)
    label = 1
    for ann in anns_by_image.get(image_info["id"], []):
        category_name = cat_by_id.get(ann["category_id"], "")
        if category_name not in SAM31_COMPARE_GT_CATEGORIES:
            continue
        mask = decode_coco_segmentation(ann["segmentation"], height, width)
        pixels = mask & (out == 0)
        if pixels.sum() < SAM31_COMPARE_MIN_AREA:
            continue
        out[pixels] = label
        label += 1
    return out


def resize_mask_to_image(mask: np.ndarray, img_rgb: np.ndarray) -> np.ndarray:
    target_h, target_w = img_rgb.shape[:2]
    if mask.shape[:2] == (target_h, target_w):
        return mask
    return cv2.resize(mask.astype(np.uint16), (target_w, target_h), interpolation=cv2.INTER_NEAREST)


def mask_boundaries(mask: np.ndarray) -> np.ndarray:
    mask = np.asarray(mask)
    boundary = np.zeros(mask.shape, dtype=bool)
    boundary[1:, :] |= mask[1:, :] != mask[:-1, :]
    boundary[:-1, :] |= mask[1:, :] != mask[:-1, :]
    boundary[:, 1:] |= mask[:, 1:] != mask[:, :-1]
    boundary[:, :-1] |= mask[:, 1:] != mask[:, :-1]
    return boundary & (mask > 0)


def draw_instance_mask(img_rgb: np.ndarray, mask: np.ndarray, color: np.ndarray, alpha: float = 0.30) -> np.ndarray:
    mask = resize_mask_to_image(mask, img_rgb)
    out = img_rgb.copy()
    overlay = img_rgb.copy()
    foreground = mask > 0
    overlay[foreground] = color
    out[mask_boundaries(mask)] = color

    for label in sorted(int(v) for v in np.unique(mask) if int(v) != 0):
        mask_bin = (mask == label).astype(np.uint8)
        contours, _ = cv2.findContours(mask_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(out, contours, -1, tuple(map(int, color)), 2)

    return cv2.addWeighted(overlay, alpha, out, 1 - alpha, 0)


def masks_to_numpy_stack(masks) -> np.ndarray:
    if hasattr(masks, "detach"):
        arr = masks.detach().float().cpu().numpy()
    else:
        arr = np.asarray(masks)
    arr = np.squeeze(arr)
    if arr.ndim == 2:
        arr = arr[None, ...]
    if arr.ndim > 3:
        arr = arr.reshape((-1,) + arr.shape[-2:])
    return arr


def scores_to_numpy(scores, n: int) -> np.ndarray:
    if scores is None:
        return np.ones(n, dtype=float)
    if hasattr(scores, "detach"):
        arr = scores.detach().float().cpu().numpy()
    else:
        arr = np.asarray(scores, dtype=float)
    arr = np.squeeze(arr).reshape(-1)
    if arr.size < n:
        arr = np.pad(arr, (0, n - arr.size), constant_values=1.0)
    return arr[:n]


def binarize_sam_mask(mask: np.ndarray) -> np.ndarray:
    if mask.dtype == bool:
        return mask
    if SAM31_COMPARE_MASK_THRESH is not None:
        return mask > SAM31_COMPARE_MASK_THRESH
    finite = mask[np.isfinite(mask)]
    if finite.size and finite.min() >= 0.0 and finite.max() <= 1.0:
        return mask > 0.5
    return mask > 0.0


def sam31_autocast_context():
    if not torch.cuda.is_available():
        return nullcontext()
    raw_dtype = os.getenv("SAM31_COMPARE_AMP_DTYPE", "bfloat16").strip().lower()
    dtype = torch.float16 if raw_dtype in {"fp16", "float16", "half"} else torch.bfloat16
    return torch.autocast(device_type="cuda", dtype=dtype)


def sam3_prediction_instance_mask(processor, pil_image: Image.Image, prompts: list[str]):
    # SAM3's eval-time ViT fast path can emit bfloat16 activations while some
    # weights remain float32. Autocast keeps linear/matmul operands compatible.
    with sam31_autocast_context():
        inference_state = processor.set_image(pil_image)
    height, width = pil_image.height, pil_image.width
    pred_mask = np.zeros((height, width), dtype=np.uint16)
    rows = []
    next_label = 1

    for prompt in prompts:
        with torch.inference_mode(), sam31_autocast_context():
            output = processor.set_text_prompt(state=inference_state, prompt=prompt)
        mask_stack = masks_to_numpy_stack(output.get("masks", []))
        scores = scores_to_numpy(output.get("scores"), len(mask_stack))
        for idx, (raw_mask, score) in enumerate(zip(mask_stack, scores)):
            if float(score) < SAM31_COMPARE_SCORE_THRESH:
                continue
            binary = binarize_sam_mask(raw_mask)
            binary = resize_mask_to_image(binary.astype(np.uint8), np.asarray(pil_image)) > 0
            pixels = binary & (pred_mask == 0)
            area = int(pixels.sum())
            if area < SAM31_COMPARE_MIN_AREA:
                continue
            pred_mask[pixels] = next_label
            rows.append({
                "label": next_label,
                "prompt": prompt,
                "sam3_index": idx,
                "score": float(score),
                "area_px": area,
            })
            next_label += 1
    return pred_mask, rows


sam3_path = str(SAM3_REPO)
if sam3_path not in sys.path:
    sys.path.insert(0, sam3_path)

from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

print("Loading SAM3 checkpoint:", SAM31_CHECKPOINT)
print("Prompts:", SAM31_COMPARE_PROMPTS)
print("GT categories:", sorted(SAM31_COMPARE_GT_CATEGORIES))
model = build_sam3_image_model(
    checkpoint_path=str(SAM31_CHECKPOINT),
    load_from_HF=False,
    eval_mode=True,
    enable_segmentation=True,
)
model.eval()
processor = Sam3Processor(model)
print("SAM31_COMPARE_AMP_DTYPE:", os.getenv("SAM31_COMPARE_AMP_DTYPE", "bfloat16"))

saved = []
instance_rows = []
for image_info in image_infos:
    image_path = COCO_ROOT / SAM31_COMPARE_SPLIT / image_info["file_name"]
    tile_id = Path(image_info["file_name"]).stem
    pil_image = Image.open(image_path).convert("RGB")
    image = np.asarray(pil_image)
    gt_mask = coco_gt_instance_mask(image_info)
    pred_mask, pred_rows = sam3_prediction_instance_mask(processor, pil_image, SAM31_COMPARE_PROMPTS)

    pred_path = PRED_MASK_DIR / f"{tile_id}_sam31_pred.png"
    Image.fromarray(pred_mask.astype(np.uint16)).save(pred_path)

    for row in pred_rows:
        row.update({"tile_id": tile_id, "pred_mask_file": str(pred_path.relative_to(OUTPUT_ROOT))})
        instance_rows.append(row)

    gt = draw_instance_mask(image, gt_mask, GT_COLOR)
    pred = draw_instance_mask(image, pred_mask, PRED_COLOR)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    panels = [image, gt, pred]
    titles = [
        f"{tile_id}: original",
        f"ground truth n={int(gt_mask.max())}",
        f"SAM3 prediction n={int(pred_mask.max())}, score>={SAM31_COMPARE_SCORE_THRESH:.2f}",
    ]
    for ax, im, title in zip(axes, panels, titles):
        ax.imshow(im)
        ax.set_title(title)
        ax.axis("off")

    handles = [
        plt.Line2D([0], [0], color=GT_COLOR / 255, lw=4, label="ground truth cell boundary"),
        plt.Line2D([0], [0], color=PRED_COLOR / 255, lw=4, label="SAM3 prediction"),
    ]
    fig.legend(handles=handles, loc="lower center", ncol=len(handles))
    fig.tight_layout(rect=[0, 0.06, 1, 1])

    out_path = COMPARE_DIR / f"{tile_id}_original_gt_sam31_pred.png"
    fig.savefig(out_path, dpi=180)
    plt.close(fig)
    saved.append(out_path)
    display(IPyImage(filename=str(out_path)))

instance_csv = COMPARE_DIR / "sam31_prediction_instances.csv"
with instance_csv.open("w", newline="") as handle:
    fieldnames = ["tile_id", "label", "prompt", "sam3_index", "score", "area_px", "pred_mask_file"]
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(instance_rows)

print("Saved comparisons to:", COMPARE_DIR)
print("Saved prediction masks to:", PRED_MASK_DIR)
print("Saved prediction instance table to:", instance_csv)
print("Files:")
for path in saved:
    print(path)
